# Day 12: Cost of fence, where the cost for each section is the peremiter multiplied by the total number of that letter

In [3]:
import numpy as np
from operator import itemgetter, attrgetter
def loadData():
    garden = []
    with open("../input/12.txt", "r") as file: 
        for line in file:
            garden.append(line.strip())
    return garden

def loadTest(test_string):
    garden = []
    for line in test_string.split("\n"):
        garden.append(line)
    return garden

test_string_1 = """RRRRIICCFF
RRRRIICCCF
VVRRRCCFFF
VVRCCCJFFF
VVVVCJJCFE
VVIVCCJJEE
VVIIICJJEE
MIIIIIJJEE
MIIISIJEEE
MMMISSJEEE"""

test_string_2 = """AAAA
BBCD
BBCC
EEEC"""

data = loadData()
test_1 = loadTest(test_string_1)
test_2 = loadTest(test_string_2)


In [11]:
def count_instances(data):
    letter_counter = {}
    for line in data:
        for letter in line:
            if letter in letter_counter.keys():
                letter_counter[letter] += 1
            else:
                letter_counter[letter] = 1
    
    return letter_counter

def create_areas(data):
    pass

class Letter:
    def __init__(self, letter: str, x: int, y: int):
        self.letter = letter
        self.parent = self
        self.children = [self]
        self.x = x
        self.y = y
        self.surface = 0
        self.corners = 0

    def __repr__(self): 
        return self.letter
    
    def compare(self, area, other_position):
        other = area[other_position[0]][other_position[1]]
        # If it is two diffirent letters, wee increase the surface for the area with 1
        if self.letter != other.letter:
            self.surface += 1
            return 1
        # If it is not already in the same set, combine the two sets
        if self.parent != other.parent:
            self.combineAreas(other)
        
        return 0

    def nrChildren(self):
        return len(self.children)

    def combineAreas(self, other):
        """ Should use parent found in compare"""
        parent = self.parent
        other_parent = other.parent
        if parent.nrChildren() > other_parent.nrChildren():
            other_parent.makeOtherParent(parent)
            return
        parent.makeOtherParent(other_parent)
        
    def makeOtherParent(self, other):
        self.parent = other
        other.addChildren(self.children)
        self.removeChildren()

    def updateParent(self, new_parent):
        self.parent = new_parent

    def addChildren(self, other_children):
        for child in other_children:
            child.updateParent(self)
        self.children.extend(other_children)

    def removeChildren(self):
        self.children = []

    def findOldestParent(self):
        if self.parent == self:
            return self
        
        oldest_parent = self.parent.findOldestParent()
        self.parent = oldest_parent
        return oldest_parent

    def findInfo(self, area: list, x_lim: int, y_lim: int) -> None:
        """Look in 4 directions for neighbours if they belong to the same area or 
        if there should be a fence"""

        if self.x == 0 :
            self.surface += 1
            left = 1
        else:
            left = self.compare(area, [self.x-1, self.y])

        if self.y == 0:
            up = 1
            self.surface += 1
        else: 
            up = self.compare(area, [self.x, self.y-1])

        if self.x == x_lim:
            right = 1
            self.surface += 1
        else: 
            right = self.compare(area, [self.x + 1, self.y])
        
        if self.y == y_lim:
            down = 1
            self.surface += 1
        else: 
            down = self.compare(area, [self.x, self.y + 1])

        # We check for all corners
        for vertical, horizontal in [(up, left), (up, right), (down, left), (down, right)]:
            self.corners += int(vertical + horizontal == 1)


    def giveArea(self) -> int:
        return self.surface * self.parent.nrChildren()
    
    def giveTestArea(self):
        return self.surface * self.parent.nrChildren(), self.letter
    
    def givesurface(self):
        return self.surface
    
    def giveArea(self):
        return self.parent.nrChildren()

    def giveChildren(self):
        return self.parent.children
    
    def giveCorners(self):
        return self.corners
    
    def part2(self) -> int:
        # Run the code only for parents. 
        if self != self.parent:
            return 0
        # In case where an item is isolated, the answer is always 4
        if self.surface == 4: 
            print(self.letter, "Returneret 4")
            return 4
        
        corners = 0
        for child in self.children:
            corners += child.giveCorners()
        print(self.letter, corners)
        return corners * self.nrChildren()

        
            
            


In [15]:
def createArea(data):
    area = []
    for x_idx, row in enumerate(data): 
        letters_in_row = []
        for y_idx, letter in enumerate(row):
            # Create new letter, and store it in the row
            letters_in_row.append(Letter(letter, x_idx, y_idx))
        area.append(letters_in_row)
    return area

def searchArea(area):
    x_lim = len(area[0]) - 1
    y_lim = len(area) - 1
    for row in area:
        for letter in row:
            letter.findInfo(area, x_lim, y_lim)
    return

def testArea(area):
    sum = []
    for row in area:
        r = []
        for letter in row:
            r.append(letter.giveArea())
        sum.append(r)
        print(r)
    print(sum)

def extractArea(area):
    total_sum = {}
    sum = 0
    for row in area:
        for letter in row: 
            val, category = letter.giveTestArea()
            sum += val
            if category in total_sum.keys():
                total_sum[category] += val
            else:
                total_sum[category] = val
    return total_sum, sum

def part2(area):
    sum = 0
    for row in area:
        for letter in row:
            sum += letter.part2()
    return sum
area = createArea(test_2)
searchArea(area)
# testArea(area)
# extractArea(area)
part2(area)
for row in area:
    print([letter.giveCorners() for letter in row])
# area
for row in test_2:
    print(row)

A 12
D Returneret 4
B 8
C 8
E 8
[2, 4, 4, 2]
[2, 2, 2, 0]
[2, 2, 2, 2]
[2, 4, 2, 2]
AAAA
BBCD
BBCC
EEEC


In [8]:
for row in test_1:
    print(row)

RRRRIICCFF
RRRRIICCCF
VVRRRCCFFF
VVRCCCJFFF
VVVVCJJCFE
VVIVCCJJEE
VVIIICJJEE
MIIIIIJJEE
MIIISIJEEE
MMMISSJEEE


In [ ]:
def run(area, pos, look, total_sides, start_position):
    if area[pos].look_up(look):
        